# Elastodynamic MHM: write the endpoint equations

Follow **meshes → spatial operators → local equations → global balance → assemble → solve → physical fields**. We use two triangular macrocells and a free-body translation accelerated by a constant force. This exactly representable patch verifies the equations; a separate smooth 3D family supplies spatial rates.

The companion contains orchestration and integrated spatial operators. Numerical Newmark stepping is the reusable `pymhm.linalg.dynamics.newmark_step`. We write the local and global endpoint equations here.

Use the locked **`introduction`** Pixi environment; the independent conforming reference executes UFL and DOLFINx.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download checksum-verified support files; extraction does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/5fea6233a056fc6eaa5697716f2b77887694d8a7a56615f4a43c3d4f8dbaa750/formulation_workflow-companion.zip"
COMPANION_SHA256 = "5fea6233a056fc6eaa5697716f2b77887694d8a7a56615f4a43c3d4f8dbaa750"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Prepare only this notebook's declared inputs, without executing its equations.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("waves/elastodynamics/formulation_workflow.ipynb", directory=ROOT)


In [ ]:
from pathlib import Path
import numpy as np
from scipy import sparse
from pymhm import Equation, LocalEquations, MultiscaleProblem, assemble
from pymhm.meshes.triangle import TriangleMesh
from pymhm.linalg.dynamics import newmark_step
from pymhm.postprocessing.nodal import nodal_field
from examples.tutorial_elastodynamic_equations import prepare, initialize, advance

mesh = TriangleMesh.unit_square()
rho, dt = 2.4, 0.02
acceleration = np.array([1.0, -0.4])
traction = {int(face): np.zeros(2) for face in mesh.boundary_faces}

def force(time, points):
    """Return physical force density rho times the exact acceleration."""
    return rho * acceleration


## 1. Specify the spatial weak forms

The isotropic operators are

$$
\begin{aligned}
 m_T(u,v)&=(\rho u,v)_T,\\
 a_T(u,v)&=2(\mu\varepsilon(u),\varepsilon(v))_T
       +(\lambda_L\operatorname{div}u,\operatorname{div}v)_T.
\end{aligned}
$$

In UFL these integrands are `rho*inner(u,v)` and `2*mu*inner(sym(grad(u)),sym(grad(v))) + lame_lambda*div(u)*div(v)`. The companion's `prepare` integrates these declared isotropic operators with common Basix quadrature/scatter kernels, builds oriented traction maps, and owns reusable factors. Here density is 2.4 and both Lamé coefficients are one. This primal material is compressible; it has no uniform incompressible-limit claim.

## 2. Write source and interface response equations

Let $u_T^f,v_T^f$ be the force-driven Newmark history without interface force. Let $U_T,V_T$ be the responses to constant slabwise negative traction $Q_T\lambda$. Then

$$
\begin{aligned}
 u_T+U_T\lambda_T&=u_T^f,\\
 -\sum_TQ_T^T u_T&=-b,\\
 v_T&=v_T^f-V_T\lambda_T.
\end{aligned}
$$

The multiplier is negative physical Cauchy traction. Inertia makes the endpoint local operator invertible; no static rigid-motion kernel is inserted. Boundary traction is converted into fixed multiplier coordinates by the prepared geometric binding.


In [ ]:
def local_endpoint(item):
    """Write u + U lambda = u_free and signed endpoint displacement moments."""
    local, free_displacement, lift = item
    return LocalEquations(
        a=sparse.eye(local.mass.shape[0]),
        L=free_displacement,
        b=lift,
        c=-local.coupling.T,
        dofs=local.trace_dofs,
        field_data=(nodal_field("displacement", local.mesh, local.degree, components=2),),
    )

def global_endpoint(boundary):
    """Impose minus the exterior displacement moments on unfixed faces."""
    return Equation(0, -boundary)


## 3. Advance the local histories, assemble and solve

Use the shared Newmark primitive with beta=1/4 and gamma=1/2. All local steps finish at the same macro time. The example has one local step per macro step. `prepare` owns the native factors until its context exits. Full traction data allow the free rigid translation.

We first advance each local history, then pass its free displacement and interface response into the equations above. Assembly resolves each global face once. Named displacement and the recovered velocity remain in the executed local basis.


In [ ]:
records = []
fields = None
with prepare(mesh, time_step=dt, degree=2, density=rho, traction=traction) as data:
    previous = initialize(data)
    for step in range(4):
        free_u, free_v = [], []
        for index, local in enumerate(data.locals):
            u, v = newmark_step(
                local.mass, local.stiffness,
                data.step_factors[index], data.mass_factors[index], dt,
                previous.displacement[index], previous.velocity[index],
                local.load_at_time(force, step * dt),
                local.load_at_time(force, (step + 1) * dt),
            )
            free_u.append(u)
            free_v.append(v)
        items = tuple(zip(data.locals, free_u, data.lifts, strict=True))
        problem = MultiscaleProblem(
            global_endpoint(data.boundary), local_endpoint, items,
            data.trace_size, (0,) * len(items), fixed=data.fixed,
        )
        system = assemble(problem)
        solution = system.solve()
        velocity = tuple(
            v - lift @ solution.trace[local.trace_dofs]
            for local, lift, v in zip(data.locals, data.velocity_lifts, free_v, strict=True)
        )
        # Independently replay the original local Newmark histories and interface balance.
        reference = advance(data, previous, force)
        for computed, exact in zip(solution.fields, reference.displacement, strict=True):
            np.testing.assert_allclose(computed, exact, rtol=1e-10, atol=1e-12)
        for computed, exact in zip(velocity, reference.velocity, strict=True):
            np.testing.assert_allclose(computed, exact, rtol=1e-10, atol=1e-12)
        assert reference.l2_error(0.5 * reference.time**2 * acceleration) < 1e-11
        assert reference.l2_error(reference.time * acceleration, velocity=True) < 1e-10
        records.append({
            "time": reference.time,
            "displacement_L2": reference.l2_error(0.5 * reference.time**2 * acceleration),
            "velocity_L2": reference.l2_error(reference.time * acceleration, velocity=True),
            "global_original_relative_residual": solution.raw_residual,
        })
        fields = solution.field("displacement")
        previous = reference
records


## 4. Plot signed displacement with the macro mesh

This constant translation should have equal values on both sides of the interior macroface. Keep separate local coefficient vectors even when this patch is continuous. A genuine heterogeneous trajectory can have distinct one-sided values.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.tri as mtri

fig, axes = plt.subplots(1, 2, figsize=(9, 3.8), layout="constrained")
for component, ax in enumerate(axes):
    for field in fields:
        local_mesh = field.mesh
        values = field.evaluate(local_mesh.points)[:, component]
        image = ax.tripcolor(mtri.Triangulation(local_mesh.points[:, 0], local_mesh.points[:, 1], local_mesh.cells),
                     values, shading="gouraud", vmin=0.5*previous.time**2*acceleration[component]-1e-5,
                     vmax=0.5*previous.time**2*acceleration[component]+1e-5)
    fig.colorbar(image, ax=ax, label="Displacement")
    ax.triplot(mesh.points[:, 0], mesh.points[:, 1], mesh.cells, color="black", linewidth=0.7)
    ax.set(xlabel="x", ylabel="y", title=f"Displacement component {component + 1}", aspect="equal")
Path("build/tutorial-methods").mkdir(parents=True, exist_ok=True)
fig.savefig("build/tutorial-methods/elastodynamic-rigid-fields.png", dpi=200, metadata={"Author":"IPES Research Group", "License":"CC BY 4.0 — https://creativecommons.org/licenses/by/4.0/"})
plt.show()


## Independent classical Galerkin reference

On global P2 vector grids of 8, 16 and 32 subdivisions, assemble exactly the same density, isotropic energy and force. Zero exterior traction is the natural boundary condition. Initial displacement and velocity are zero. Use the common Newmark primitive and solve its positive inertial operator; no static rigid gauge is inserted. The exact rigid translation lies in every reference space, so these controls measure reproduction rather than a spatial rate.


In [ ]:
import ufl
from dolfinx import fem, mesh as native_mesh
from mpi4py import MPI
from pymhm import compile_form
from pymhm.linalg.linear import factorize

classical = []
for resolution in (8, 16, 32):
    domain = native_mesh.create_unit_square(MPI.COMM_SELF, resolution, resolution)
    space = fem.functionspace(domain, ("Lagrange", 2, (2,)))
    u, v = ufl.TrialFunction(space), ufl.TestFunction(space)
    dx = ufl.dx(domain=domain)
    mass_form = rho * ufl.inner(u,v)*dx
    stiffness_form = (2*ufl.inner(ufl.sym(ufl.grad(u)),ufl.sym(ufl.grad(v)))
                      + ufl.div(u)*ufl.div(v))*dx
    load_form = rho * ufl.inner(ufl.as_vector(tuple(acceleration)),v)*dx
    size = 2 * len(space.tabulate_dof_coordinates())
    M, K = compile_form(mass_form, (size,size)), compile_form(stiffness_form, (size,size))
    F = compile_form(load_form, (size,))
    displacement, velocity = np.zeros(size), np.zeros(size)
    with factorize(M) as mass_factor, factorize(M + dt**2/4*K) as step_factor:
        for step in range(4):
            displacement, velocity = newmark_step(M,K,step_factor,mass_factor,dt,
                                                   displacement,velocity,F,F)
    reference = fem.Function(space)
    reference.x.array[:] = displacement
    exact = ufl.as_vector(tuple(0.5*(4*dt)**2 * acceleration))
    difference = reference-exact
    error = np.sqrt(fem.assemble_scalar(fem.form(ufl.inner(difference,difference)*dx)))
    assert error < 1e-10
    classical.append({"resolution":resolution,"displacement_L2":float(error)})
classical


## 5. Separate spatial and temporal rates

The translation patch above verifies the endpoint equations, but has no approximation error to measure. We now solve a smooth **nonconstant** two-dimensional problem with zero initial and exterior displacement. Its spaces are local $P_3$ displacement on one triangle and vector $P_1$ negative traction. An independently assembled conforming $P_3$ reference supplies a classical physical control. The spatial and Newmark temporal studies remain separate.

The original elastodynamic analysis predicts displacement and velocity $L^2$ order $\ell+2$ and broken strain/stress order $\ell+1$ for smooth fields, compatible local lifting, resolved time error and $P_\ell$ traction. Here $\ell=1$. This new 2D analytical case verifies those estimates on its declared spaces; it is not a reproduction of the paper's three-dimensional numerical table. [Gomes et al. (2017)](https://doi.org/10.20906/CPS/CILAMCE2017-0399).


## 6. Derive a nonconstant dynamic source

With $\rho=\lambda_L=\mu=1$, define

$$
\begin{aligned}
w(x,y)&=(\sin(\pi x)\sin(\pi y),\sin(2\pi x)\sin(\pi y)),\\
u_*(t,x,y)&=\tfrac12t^2w(x,y),\qquad v_*(t,x,y)=tw(x,y),\\
\sigma(w)&=2\varepsilon(w)+(\nabla\cdot w)I,\\
f(t,x,y)&=w(x,y)-\tfrac12t^2\nabla\cdot\sigma(w).
\end{aligned}
$$

The initial displacement and velocity vanish, and the entire exterior has zero displacement. Quadratic time dependence reduces time-truncation pollution without changing the spatial interface method. A finer time-step control still checks the discrete march. First write the continuous stress with UFL; then supply its independently differentiated NumPy force to the local history.


In [ ]:
def shape_w(points):
    """Evaluate the two independent sine-product displacement components."""
    x, y = points.T
    return np.column_stack((np.sin(np.pi*x)*np.sin(np.pi*y),
                            np.sin(2*np.pi*x)*np.sin(np.pi*y)))


def dynamic_force(time, points):
    """Evaluate w-t²/2 div(sigma(w)), from analytic physical derivatives."""
    x, y = points.T
    w = shape_w(points)
    elastic_force = np.pi**2*np.column_stack((
        4*w[:, 0]-4*np.cos(2*np.pi*x)*np.cos(np.pi*y),
        7*w[:, 1]-2*np.cos(np.pi*x)*np.cos(np.pi*y),
    ))
    return w+0.5*time*time*elastic_force


def symbolic_dynamic_data(domain):
    """Independently differentiate the continuous isotropic Cauchy stress in UFL."""
    x = ufl.SpatialCoordinate(domain)
    w = ufl.as_vector((ufl.sin(np.pi*x[0])*ufl.sin(np.pi*x[1]),
                       ufl.sin(2*np.pi*x[0])*ufl.sin(np.pi*x[1])))
    sigma = 2*ufl.sym(ufl.grad(w))+ufl.div(w)*ufl.Identity(2)
    return w, sigma, -ufl.div(sigma)


## 7. Prepare the same endpoint formulation on a coarse mesh

`prepare` integrates the mass, energy and negative-traction operators displayed above. `advance` is the convenience form of the local-history → global-endpoint → reconstruction loop already written in Section 3, including an independent replay of the original local Newmark histories. A source change only changes physical load vectors. Each local coefficient field retains its own value on macrofaces.


In [ ]:
from threadpoolctl import threadpool_limits
from pymhm import FaceSpace, SkeletonSpace

wave_macro = TriangleMesh.unit_square(8)
wave_skeleton = SkeletonSpace(
    wave_macro, tuple(FaceSpace.uniform(1) for _ in wave_macro.faces), components=2,
)
wave_dt, wave_final = 0.005, 0.1
with threadpool_limits(1), prepare(
    wave_macro, skeleton=wave_skeleton, time_step=wave_dt,
    degree=3, local_refinement=1, density=1.0,
    lame_lambda=1.0, lame_mu=1.0, quadrature_order=10,
) as wave_data:
    wave_state = initialize(wave_data)
    for _ in range(round(wave_final/wave_dt)):
        wave_state = advance(wave_data, wave_state, dynamic_force)
print({
    "time":wave_state.time,
    "displacement_L2":wave_state.l2_error(lambda p:0.5*wave_final**2*shape_w(p),order=14),
    "velocity_L2":wave_state.l2_error(lambda p:wave_final*shape_w(p),velocity=True,order=14),
    "endpoint_moment_residual":wave_state.constraint_residual,
})


## 8. Independently assemble a classical conforming reference

The global $P_3$ Galerkin solve uses the same mass, isotropic Cauchy stress, force and strong homogeneous displacement data. Its UFL divergence derives the force separately from `dynamic_force`. It has no MHM trace or local lifting. The archived spatial reference uses meshes $16,32,64$ and verifies its own analytical displacement, velocity and stress errors before the finest field is used as a baseline.


In [ ]:
import basix
import basix.ufl
from pymhm.backends.spaces import bind_space, coefficient_map
from pymhm.fem.scalar.triangle import nodal_space

reference_mesh = TriangleMesh.unit_square(64)
reference_space = bind_space(reference_mesh, basix.ufl.element(
    "Lagrange", "triangle", 3, shape=(2,),
    lagrange_variant=basix.LagrangeVariant.equispaced,
))
try:
    u, v = ufl.TrialFunction(reference_space.space), ufl.TestFunction(reference_space.space)
    domain = reference_space.space.mesh
    w, sigma_w, elastic_force = symbolic_dynamic_data(domain)
    dx = ufl.Measure("dx",domain=domain,metadata={"quadrature_degree":14})
    mass = compile_form(ufl.inner(u,v)*dx)
    stiffness = compile_form((2*ufl.inner(ufl.sym(ufl.grad(u)),ufl.sym(ufl.grad(v)))
                              +ufl.div(u)*ufl.div(v))*dx)
    force0 = compile_form(ufl.inner(w,v)*dx)
    force1 = compile_form(ufl.inner(elastic_force,v)*dx)
    _, nodes = nodal_space(reference_mesh,3)
    exterior = np.flatnonzero(np.any(np.isclose(nodes,0)|np.isclose(nodes,1),axis=1))
    fixed = coefficient_map(reference_space)[(2*exterior[:,None]+np.arange(2)).ravel()]
    free = np.setdiff1d(np.arange(reference_space.size),fixed)
    m, k = mass[free][:,free], stiffness[free][:,free]
    reference_u, reference_v = np.zeros(len(free)), np.zeros(len(free))
    with threadpool_limits(1), factorize(m) as mf, factorize(m+wave_dt**2/4*k) as sf:
        for step in range(round(wave_final/wave_dt)):
            old_force = force0[free]+0.5*(step*wave_dt)**2*force1[free]
            new_force = force0[free]+0.5*((step+1)*wave_dt)**2*force1[free]
            reference_u, reference_v = newmark_step(
                m,k,sf,mf,wave_dt,reference_u,reference_v,old_force,new_force,
            )
    reference_coefficients = np.zeros(reference_space.size)
    reference_coefficients[free] = reference_u
    # Sample each macrocell separately; never average incident MHM values.
    sample = TriangleMesh.unit_square(4)
    bary = np.column_stack((1-sample.points.sum(axis=1),sample.points))
    inside = bary[:,0]>=-1e-14
    bary = bary[inside]
    sample_triangles = mtri.Triangulation(bary[:,1],bary[:,2]).triangles
    coordinates, triangles, numerical, exact, classical_values, velocities = [], [], [], [], [], []
    offset = 0
    for cell in range(len(wave_macro.cells)):
        points = bary@wave_macro.points[wave_macro.cells[cell]]
        values, local_velocity = wave_state.evaluate(cell,bary)
        coordinates.append(points);triangles.append(sample_triangles+offset)
        numerical.append(values[0]);velocities.append(local_velocity[0]);exact.append(0.5*wave_final**2*shape_w(points))
        classical_values.append(reference_space.evaluate(reference_coefficients,points))
        offset += len(points)
    points = np.concatenate(coordinates);triangles = np.concatenate(triangles)
    uh, ue, uc = np.concatenate(numerical), np.concatenate(exact), np.concatenate(classical_values)
    vh = np.concatenate(velocities)
finally:
    reference_space.close()


In [ ]:
from examples.introduction.vector import plot_field_panels

panels = {
    "Analytical displacement x":(points,triangles,ue[:,0]),
    "MHM displacement x":(points,triangles,uh[:,0]),
    "Conforming P3 displacement x":(points,triangles,uc[:,0]),
    "MHM displacement error x":(points,triangles,uh[:,0]-ue[:,0]),
    "Analytical velocity magnitude":(points,triangles,np.linalg.norm(wave_final*shape_w(points),axis=1)),
    "MHM velocity magnitude":(points,triangles,np.linalg.norm(vh,axis=1)),
}
figure = plot_field_panels(wave_macro,panels,figsize=(13,8))
output = ROOT/"build/tutorial-methods"
output.mkdir(parents=True,exist_ok=True)
figure.savefig(output/"elastodynamics-smooth-fields.png",dpi=200,
               metadata={"Author":"IPES Research Group","License":"CC BY 4.0"})
plt.show()


## 9. Read qualified spatial and temporal sequences

Keep physical displacement, velocity and raw Cauchy stress separate. Spatial curves compare directly to the analytical fields at the same physical time. Temporal curves compare the same fixed spatial scheme against its separately refined time reference. A second-order time result does not establish a spatial multiscale rate. The figures retain every level, report every consecutive rate using the actual mesh ratios and show normalized target amplitudes over the final four levels.


In [ ]:
import json
from examples.tutorial_convergence import refinement_series,plot_method_series,asymptotic_summary
record_path=ROOT/'examples/results/tutorial-methods/elastodynamics-spatial-asymptotic.json'
record=json.loads(record_path.read_text())
spatial=refinement_series(
    record_path,record['rows'],'H',
    {'displacement L2':'displacement_l2','velocity L2':'velocity_l2','physical stress L2':'stress_l2'},
    {'displacement L2':3.0,'velocity L2':3.0,'physical stress L2':2.0},method='elastodynamics',
    spaces='Smooth 2D analogue, P3/r1 locals, P1 negative traction, quadratic time dependence; endpoint norms',
    rate_provenance='Gomes et al. (2017), Figure3: published displacement and velocity L2 order3; broken gradient order2',
    root=ROOT,
)
record_path=ROOT/'examples/results/elastodynamics/comparison.json'
record=json.loads(record_path.read_text())
rows=[dict(dt=row['dt'],**row['difference_to_time_reference']) for row in record['temporal']]
temporal=refinement_series(
    record_path,rows,'dt',
    {'displacement L2 difference':'displacement_l2','velocity L2 difference':'velocity_l2'},
    {'displacement L2 difference':2.0,'velocity L2 difference':2.0},method='elastodynamics-time',
    spaces='Fixed spatial space, Newmark beta1/4 gamma1/2, independently refined time reference',
    rate_provenance='Second-order time integration; differences to a numerical time reference, not exact errors',
    refinement='time step',root=ROOT,
)
from IPython.display import SVG, display

plot_directory = Path("build/tutorial-methods")
plot_directory.mkdir(parents=True, exist_ok=True)
for series in (spatial, temporal):
    print(series["method"], asymptotic_summary(series))
    figure = plot_method_series(series)
    figure_path = plot_directory / f"{series['method']}-convergence.svg"
    figure.savefig(
        figure_path,
        metadata={"Creator": "IPES Research Group", "Rights": "CC BY 4.0"},
    )
    display(SVG(filename=str(figure_path)))
    plt.close(figure)
